# Embedding

In [ ]:
import torch as t
from torch import Tensor
from jaxtyping import Int, Float
import torch.nn as nn
from dataclasses import dataclass

@dataclass
class Config:
    d_model: int = 768 # This is the n_embd
    debug: bool = True
    layer_norm_eps: float = 1e-5
    d_vocab: int = 50257
    init_range: float = 0.02
    n_ctx: int = 1024
    d_head: int = 64
    d_mlp: int = 3072
    n_heads: int = 12
    n_layers: int = 12


cfg = Config()
### To access the Config
print("number of attention heads: ", cfg.d_head)
print(cfg)

number of attention heads:  64
Config(d_model=768, debug=True, layer_norm_eps=1e-05, d_vocab=50257, init_range=0.02, n_ctx=1024, d_head=64, d_mlp=3072, n_heads=12, n_layers=12)


### RECALL

```python 
class Embedding(nn.Module):
    def __init__(self, vocab_size, n_embd):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd)
    
    def forward(self, idx):
        return self.wte(idx)
```

We now go one layer deeper and manually create ``self.wte`` and via index produce our token embedding


In [ ]:
class Embed(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self.we = nn.Parameter(t.empty(config.d_vocab, config.d_model))
        nn.init.normal(self.we, std=config.init_range)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_model"]:
        return self.we[tokens]

### Positional Embedding

### RECALL

```python
class PosEmbed(nn.Module):
    def __init__(self, block_size, n_embd):
        super().__init__()
        self.wpe = nn.Embedding(block_size, n_embd)
    
    def forward(self, tokens):
        B, T = tokens.shape
        pos = t.arange(T)
        pos_emb = self.wpe(pos)
        return pos_emb
```

In [ ]:
class PosEmbed(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self.wpe = nn.Parameter(t.empty(config.n_ctx, config.d_model))
        nn.init.normal_(self.wpe, std=config.init_range)

    def forward(self, tokens):
        B, T = tokens.shape
        pos = t.arange(T)
        pos_emb = self.wpe[tokens]
        return pos_emb.unsqueeze(0).expand(B, -1, -1)

### LayerNorm 

In [2]:
class LayerNorm(nn.Module):
    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self.w = nn.Parameter(t.ones(config.d_model))
        self.b = nn.Parameter(t.zeros(config.d_model))

    def forward(self,residual: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        mean = residual.mean(dim=-1, keepdim=True)
        std = (residual.var(dim=-1, keepdim=True, unbiased=False) + self.config.layer_norm_eps).sqrt()
        res_normed = ((residual - mean)/std)*self.w + self.b
        return res_normed